# GCS kitchen navigation and pickup

In [1]:
# %% Collapsed demo infrastructure
import math
import threading
import time
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import rclpy
from rclpy.node import Node
from scipy.spatial.transform import Rotation

import coraplex
from coraplex.datastructures.dataclasses import Context
from coraplex.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.execution_environment import simulated_robot
from coraplex.locations.factories import reachability_location
from coraplex.plans.factories import code, sequential
from coraplex.robot_plans.actions.composite.tool_based import PouringAction
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.robot_plans.actions.core.placing import PlaceAction
from coraplex.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from coraplex.testing import setup_world
from coraplex.view_manager import ViewManager
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.datastructures.definitions import TorsoState
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Bowl, Milk, PouringCup, Table
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Point3, Pose
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.geometry import VolumetricBoundingBox
from semantic_digital_twin.world_description.graph_of_convex_sets.base import GraphOfConvexSets
from semantic_digital_twin.world_description.graph_of_convex_sets.boxes import VolumetricGraphOfBoundingBoxes
from semantic_digital_twin.world_description.shape_collection import BoundingBoxCollection
from semantic_digital_twin.world_description.world_entity import Body


@dataclass(frozen=True)
class FixedKitchen:
    """Scene state shared by fixed-kitchen demonstrations."""

    world: World
    """Apartment world used by the demonstration."""

    robot: PR2
    """Robot annotation bound to the apartment world."""

    initial_base_pose: Pose
    """Robot base pose before preparation or navigation."""


@dataclass(frozen=True)
class NavigationRoute:
    """Reachable base pose and the GCS path leading to it."""

    base_pose: Pose
    """Reachable robot base pose at the route destination."""

    navigation_point: Point3
    """Destination projected onto the navigation layer."""

    path: list[Point3]
    """Filtered GCS path from the preceding route endpoint."""


def setup_fixed_kitchen(
    initial_base_position: tuple[float, float] | None = None,
) -> FixedKitchen:
    """Create the fixed apartment and bind its PR2 annotation."""
    world = setup_world()
    robot = PR2.from_world(world)
    if initial_base_position is not None:
        position_x, position_y = initial_base_position
        with world.modify_world():
            robot.root.parent_connection.origin = (
                HomogeneousTransformationMatrix.from_xyz_rpy(
                    x=position_x,
                    y=position_y,
                    reference_frame=world.root,
                )
            )
    return FixedKitchen(
        world=world,
        robot=robot,
        initial_base_pose=robot.root.global_pose,
    )


def start_ros_node(node_name: str) -> tuple[Node, threading.Thread]:
    """Start a ROS node and retain its background spinning thread."""
    if not rclpy.ok():
        rclpy.init()
    ros_node = rclpy.create_node(node_name)
    ros_thread = threading.Thread(target=rclpy.spin, args=(ros_node,), daemon=True)
    ros_thread.start()
    return ros_node, ros_thread


def start_visualization_publisher(
    world: World,
    ros_node: Node,
) -> tuple[TFPublisher, VizMarkerPublisher]:
    """Start synchronized transform and marker publication for a world."""
    tf_publisher = TFPublisher(_world=world, node=ros_node)
    visualization_publisher = VizMarkerPublisher(_world=world, node=ros_node)
    return tf_publisher, visualization_publisher


def create_navigation_graph(
    kitchen: FixedKitchen,
    bounds: tuple[float, float, float, float],
    region_name: str,
    navigation_height: float = 0.2,
    obstacle_bloat: float = 0.22,
) -> GraphOfConvexSets:
    """Build and expose the fixed kitchen's GCS navigation free space."""
    minimum_x, maximum_x, minimum_y, maximum_y = bounds
    search_space = BoundingBoxCollection(
        [
            VolumetricBoundingBox(
                min_x=minimum_x,
                max_x=maximum_x,
                min_y=minimum_y,
                max_y=maximum_y,
                min_z=navigation_height - 0.05,
                max_z=navigation_height + 0.05,
                origin=HomogeneousTransformationMatrix(reference_frame=kitchen.world.root),
            )
        ],
        reference_frame=kitchen.world.root,
    )
    navigation_graph = VolumetricGraphOfBoundingBoxes.free_space_from_world(
        world=kitchen.world,
        search_space=search_space,
        bloat_obstacles=obstacle_bloat,
    )
    # create_as_region moved to HasSupportingSurface.spawn_bounding_boxes_as_region
    # upstream; dropped here since nothing downstream consumes the spawned Region --
    # navigation_graph itself (path_from_to / plot_free_space) is what's actually used.
    return navigation_graph


def navigation_point(
    base_pose: Pose,
    reference_frame: Body,
    navigation_height: float = 0.2,
) -> Point3:
    """Project a robot base pose onto the GCS navigation layer."""
    return Point3(
        x=base_pose.x,
        y=base_pose.y,
        z=navigation_height,
        reference_frame=reference_frame,
    )


def filter_navigation_path(
    raw_path: list[Point3],
    minimum_waypoint_distance: float = 0.08,
) -> list[Point3]:
    """Remove consecutive decomposition waypoints below the requested distance."""
    filtered_path = [raw_path[0]]
    for candidate in raw_path[1:-1]:
        previous = filtered_path[-1]
        distance = math.hypot(
            float(candidate.x - previous.x),
            float(candidate.y - previous.y),
        )
        if distance >= minimum_waypoint_distance:
            filtered_path.append(candidate)
    filtered_path.append(raw_path[-1])
    return filtered_path


def create_navigation_route(
    navigation_graph: GraphOfConvexSets,
    start_point: Point3,
    destination_point: Point3,
    base_pose: Pose,
    minimum_waypoint_count: int = 1,
) -> NavigationRoute:
    """Create a filtered route to a fixed, known navigation target."""
    raw_path = navigation_graph.path_from_to(start_point, destination_point)
    if raw_path is None or len(raw_path) < minimum_waypoint_count:
        raise RuntimeError("The fixed navigation target is not connected")
    return NavigationRoute(
        base_pose=base_pose,
        navigation_point=destination_point,
        path=filter_navigation_path(raw_path),
    )


def find_optional_reachable_route(
    navigation_graph: GraphOfConvexSets,
    start_point: Point3,
    target: Body | Pose,
    kitchen: FixedKitchen,
    context: Context,
    arm: Arms,
    grasp_description: GraspDescription,
    navigation_height: float = 0.2,
) -> NavigationRoute | None:
    """Return the first connected reachable target pose, when one exists."""
    for candidate_base_pose in reachability_location(
        target,
        context,
        arm,
        grasp_description,
    ):
        candidate_point = navigation_point(
            candidate_base_pose,
            kitchen.world.root,
            navigation_height,
        )
        candidate_path = navigation_graph.path_from_to(start_point, candidate_point)
        if candidate_path is not None:
            return NavigationRoute(
                base_pose=candidate_base_pose,
                navigation_point=candidate_point,
                path=filter_navigation_path(candidate_path),
            )
    return None


def find_reachable_route(
    navigation_graph: GraphOfConvexSets,
    start_point: Point3,
    target: Body | Pose,
    kitchen: FixedKitchen,
    context: Context,
    arm: Arms,
    grasp_description: GraspDescription,
    navigation_height: float = 0.2,
) -> NavigationRoute:
    """Return a connected reachable target pose or report that none exists."""
    route = find_optional_reachable_route(
        navigation_graph,
        start_point,
        target,
        kitchen,
        context,
        arm,
        grasp_description,
        navigation_height,
    )
    if route is None:
        raise RuntimeError(
            "No reachable target pose is connected to the navigation graph"
        )
    return route


def interpolate_navigation_path(
    path: list[Point3],
    final_yaw: float,
    reference_frame: Body,
    step_distance: float = 0.08,
) -> list[Pose]:
    """Convert a GCS path into closely spaced robot base poses."""
    poses = []
    for segment_start, segment_end in zip(path, path[1:]):
        start_x = float(segment_start.x)
        start_y = float(segment_start.y)
        distance_x = float(segment_end.x) - start_x
        distance_y = float(segment_end.y) - start_y
        segment_distance = math.hypot(distance_x, distance_y)
        step_count = max(1, math.ceil(segment_distance / step_distance))
        segment_yaw = math.atan2(distance_y, distance_x)
        for step_index in range(1, step_count + 1):
            progress = step_index / step_count
            poses.append(
                Pose.from_xyz_rpy(
                    x=start_x + distance_x * progress,
                    y=start_y + distance_y * progress,
                    z=0.0,
                    yaw=segment_yaw,
                    reference_frame=reference_frame,
                )
            )
    final_point = path[-1]
    poses[-1] = Pose.from_xyz_rpy(
        x=final_point.x,
        y=final_point.y,
        z=0.0,
        yaw=final_yaw,
        reference_frame=reference_frame,
    )
    return poses


def create_navigation_playback(
    route: NavigationRoute,
    reference_frame: Body,
    seconds_per_step: float = 0.08,
) -> tuple[list[Pose], list[object]]:
    """Create visible navigation poses and their sequential plan nodes."""
    final_yaw = float(route.base_pose.to_quaternion().to_rpy()[2])
    poses = interpolate_navigation_path(route.path, final_yaw, reference_frame)
    steps = []
    for pose in poses:
        steps.extend(
            [
                NavigateAction(pose),
                code(partial(time.sleep, seconds_per_step)),
            ]
        )
    return poses, steps


def plot_navigation_routes(
    navigation_graph: GraphOfConvexSets,
    routes: list[tuple[str, NavigationRoute, str]],
    target: Point3 | None = None,
    navigation_height: float = 0.2,
) -> go.Figure:
    """Plot the GCS free space and named routes for notebook inspection."""
    figure = go.Figure(navigation_graph.plot_free_space())
    for route_index, (name, route, color) in enumerate(routes):
        figure.add_trace(
            go.Scatter3d(
                x=[point.x for point in route.path],
                y=[point.y for point in route.path],
                z=[navigation_height + route_index * 0.01 for point in route.path],
                mode="lines+markers",
                name=name,
                line={"color": color, "width": 8},
            )
        )
    if target is not None:
        figure.add_trace(
            go.Scatter3d(
                x=[target.x],
                y=[target.y],
                z=[target.z],
                mode="markers",
                name="Task target",
                marker={"color": "green", "size": 8},
            )
        )
    figure.update_layout(height=600, scene_aspectmode="data")
    return figure


def front_grasp(kitchen: FixedKitchen, arm: Arms) -> GraspDescription:
    """Create the common front grasp used by fixed-kitchen examples."""
    return GraspDescription(
        approach_direction=ApproachDirection.FRONT,
        vertical_alignment=VerticalAlignment.NoAlignment,
        end_effector=ViewManager.get_end_effector_view(arm, kitchen.robot),
    )


def placement_pose_from_surface_sample(
    sampled_point: Point3,
    table: Table,
    object_body: Body,
    world: World,
) -> Pose:
    """Create an object pose from a sampled free table-surface point."""
    sampled_world_point = world.transform(sampled_point, world.root)
    table_top_height = float(table.root.global_pose.z + table.min_max_points[1].z)
    object_half_height = float(object_body.combined_mesh.extents[2] / 2)
    orientation = world.transform(
        Pose(reference_frame=table.supporting_surface),
        world.root,
    ).to_quaternion()
    return Pose.from_xyz_quaternion(
        sampled_world_point.x,
        sampled_world_point.y,
        table_top_height + object_half_height,
        *orientation.to_list(),
        reference_frame=world.root,
    )


In [2]:
kitchen = setup_fixed_kitchen(initial_base_position=(3.5, 2.0))
world, robot = kitchen.world, kitchen.robot
milk = world.get_body_by_name("milk.stl")
cereal = world.get_body_by_name("breakfast_cereal.stl")
milk_annotation = world.get_semantic_annotations_by_type(Milk)[0]

{
    "robot": tuple(round(float(value), 2) for value in robot.root.global_pose.to_position().to_list()[:3]),
    "milk": tuple(round(float(value), 2) for value in milk.global_pose.to_position().to_list()[:3]),
    "cereal": tuple(round(float(value), 2) for value in cereal.global_pose.to_position().to_list()[:3]),
}


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


{'robot': (3.5, 2.0, 0.0),
 'milk': (2.37, 2.0, 1.05),
 'cereal': (2.37, 1.8, 1.05)}

In [3]:
ros_node, ros_thread = start_ros_node("coraplex_gcs_kitchen")
context = Context(world=world, robot=robot, ros_node=ros_node)


In [11]:
tf_publisher, visualization_publisher = start_visualization_publisher(
    world,
    ros_node,
)
print(f"RViz fixed frame: {world.root.name}")
print(f"MarkerArray topic: {visualization_publisher.topic_name}")


RViz fixed frame: apartment/apartment_root
MarkerArray topic: /semworld/viz_marker


In [5]:
assert context.evaluate_conditions


In [6]:
navigation_graph = create_navigation_graph(
    kitchen,
    bounds=(0.7, 4.2, 0.2, 4.8),
    region_name="kitchen_navigation_free_space",
)
start_point = navigation_point(kitchen.initial_base_pose, world.root)
pickup_position = Point3(1.65, 2.0, 0.2, reference_frame=world.root)
pickup_yaw = math.atan2(
    float(milk.global_pose.y - pickup_position.y),
    float(milk.global_pose.x - pickup_position.x),
)
pickup_base_pose = Pose.from_xyz_rpy(
    x=pickup_position.x,
    y=pickup_position.y,
    yaw=pickup_yaw,
    reference_frame=world.root,
)
pickup_route = create_navigation_route(
    navigation_graph,
    start_point,
    pickup_position,
    pickup_base_pose,
    minimum_waypoint_count=3,
)
grasp_description = front_grasp(kitchen, Arms.LEFT)

{
    "graph_nodes": len(navigation_graph.graph.nodes()),
    "graph_edges": len(navigation_graph.graph.edges()),
    "filtered_path": [
        (round(float(point.x), 3), round(float(point.y), 3))
        for point in pickup_route.path
    ],
}


INFO:graph_of_convex_sets.py::543 free_space_from_semantic_annotation Free space calculated in 21.663033 ms
INFO:graph_of_convex_sets.py::561 free_space_from_semantic_annotation Connectivity calculated in 18.292118 ms


{'graph_nodes': 72,
 'graph_edges': 72,
 'filtered_path': [(3.5, 2.0),
  (3.139, 0.636),
  (3.039, 0.635),
  (2.097, 0.635),
  (1.65, 2.0)]}

In [7]:
plot_navigation_routes(
    navigation_graph,
    [("GCS path", pickup_route, "black")],
)


In [8]:
navigation_poses, navigation_steps = create_navigation_playback(
    pickup_route,
    world.root,
)
len(navigation_poses)


50

In [9]:
task = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        *navigation_steps,
        PickUpAction(milk_annotation, Arms.LEFT, grasp_description),
    ],
    context=context,
)

with simulated_robot:
    task.perform()


In [ ]:
left_end_effector = ViewManager.get_end_effector_view(Arms.LEFT, robot)
picked_up = world.get_connection(left_end_effector.tool_frame, milk) is not None
assert picked_up

{
    "graph_nodes": len(navigation_graph.graph.nodes()),
    "graph_edges": len(navigation_graph.graph.edges()),
    "filtered_waypoints": len(pickup_route.path),
    "navigation_updates": len(navigation_poses),
    "picked_up": picked_up,
    "cereal_remains_on_counter": cereal.parent_connection.parent == world.root,
}
